# Chaos in the three-body problem

**Burrau's Pythagorean problem**: masses 3, 4 and 5 start at rest at the corners of a 3-4-5 right triangle
(each mass opposite the side of its own length), $G = 1$. After a long dance of close encounters the system breaks up:
the lightest body is ejected and the other two leave as a binary. The encounters are very close (separations below
$10^{-3}$), so this is a job for adaptive time steps.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import physim as ps

In [ ]:
def pythagorean(dx=0.0):
    w = ps.World()
    w.add_particle([1 + dx, 3, 0], mass=3)
    w.add_particle([-2, -1, 0], mass=4)
    w.add_particle([1, -1, 0], mass=5)
    w.add_force(ps.NewtonianGravity(G=1.0))
    return w

w = pythagorean()
e0 = w.total_energy()
tr = w.run_adaptive(70.0, rtol=1e-13, atol=1e-15)
info = tr.metadata["adaptive"]
sep = np.linalg.norm(tr.pos[:, 1] - tr.pos[:, 2], axis=1)
print(f"{info['accepted']} adaptive steps; closest approach of the 4-5 pair {sep.min():.1e}; "
      f"energy error {abs(w.total_energy() / e0 - 1):.1e}")
ax = ps.plot.orbits(tr, labels=["m = 3", "m = 4", "m = 5"], lw=0.7, figsize=(6, 6))
ax.set_xlim(-4, 4); ax.set_ylim(-6, 6);

**Sensitivity to initial conditions.** Moving the first body by $10^{-9}$ gives a second solution that agrees at
first and then separates exponentially: by $t \approx 60$ the two runs are completely different. The figure-eight
orbit, by contrast, is linearly stable: the same perturbation stays small.

In [ ]:
times = np.linspace(0, 70, 1401)
a = pythagorean().run_adaptive(70, times=times, rtol=1e-13, atol=1e-15)
b = pythagorean(1e-9).run_adaptive(70, times=times, rtol=1e-13, atol=1e-15)

def eight(dx=0.0):
    w = ps.scenarios.figure_eight()
    pos = w.positions; pos[0, 0] += dx; w.positions = pos
    return w

c = eight().run_adaptive(70, times=times, rtol=1e-13, atol=1e-15)
d = eight(1e-9).run_adaptive(70, times=times, rtol=1e-13, atol=1e-15)

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.semilogy(times, np.linalg.norm(a.pos - b.pos, axis=(1, 2)), color=ps.plot.color(1), label="Pythagorean problem")
ax.semilogy(times, np.linalg.norm(c.pos - d.pos, axis=(1, 2)), color=ps.plot.color(0), label="figure-eight")
ax.set_xlabel("t"); ax.set_ylabel("distance between the two runs"); ps.plot.tidy(ax, grid=True);

In [ ]:
# For regular motion the finite-time Lyapunov estimate decays like ln(t)/t and MEGNO tends to 2.
for t_end in (70, 700):
    out = eight().lyapunov(1e-2, int(t_end / 1e-2), n=1, record_every=int(t_end / 1e-2))
    print(f"figure-eight to t = {t_end}: Lyapunov estimate {out['exponents'][0]:.4f} "
          f"(ln t / t = {np.log(t_end) / t_end:.4f}), MEGNO {out['mean_megno'][-1]:.2f}")